**ANOMALY DETECTION USING ISOLATION FOREST**

**Step-1 : Install Libraries**

In [30]:
#install gradio
!pip install gradio -q

**Step-2 : Import Libraries**

In [31]:
#import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gradio as gr

from sklearn.ensemble import IsolationForest

**Step-3 : Create Sample Dataset**

In [32]:
np.random.seed(42)

# Create Normal Transaction Data
normal_amount = np.random.normal(100, 20, 200)
normal_frequency = np.random.normal(10, 2, 200)

# Create Anomely Trasaction Data
anomaly_amount = np.array([400, 450, 500, 550, 600])
anomaly_frequency = np.array([25, 30, 35, 40, 45])


#Combine Normal and Anomely Amount
amount = np.concatenate([
    normal_amount,
    anomaly_amount
])

#Combine Normal and Anomely Frequency
frequency = np.concatenate([
    normal_frequency,
    anomaly_frequency
])

# Create DataFrame
data = pd.DataFrame({
    "Transaction_Amount" :amount,
    "Transaction_Frequency" :frequency
})

**Step-4 : Select Feature**

In [33]:
X = data[["Transaction_Amount","Transaction_Frequency"]]

**Step-5 : Create Isolation Forest**

In [34]:
model = IsolationForest(
    contamination = 0.05,
    random_state = 10
)

#Train the Model
model.fit(X)

#Predict Anomalies
predictions = model.predict(X)

# Isolation Forest
# 1 = Normal
# -1 = Anomaly

data["Anomaly"] = predictions

# Convert prediction into reable labels
data["Label"] = data["Anomaly"].map({
    1 : "Normal",
    -1 : "Anomaly"
})



**Step-6 : Create Visualization Function**

In [38]:
def create_plot():
  normal = data[data["Anomaly"] == 1]
  anomalies = data[data['Anomaly'] == -1]

  fig, ax = plt.subplots(figsize = (8, 6))

  #Plot normal data
  ax.scatter(
      normal["Transaction_Amount"],
      normal["Transaction_Frequency"],
      label = "Normal"
  )

  #Plot anomaly data
  ax.scatter(
      anomalies["Transaction_Amount"],
      anomalies["Transaction_Frequency"],
      marker = "X",
      s = 100,
      label = "Anomaly"
  )

  ax.set_xlabel("Transaction Amount")
  ax.set_ylabel("Transaction Frequency")
  ax.set_title("Isolation Forest Anomaly Detection")
  ax.legend()

  return fig

**Step-7 : Create Prediction Function for Gradio**

In [39]:
def detect_anomaly(amount, frequency):
  input_data = pd.DataFrame({
      "Transaction_Amount" : [amount],
      "Transaction_Frequency" : [frequency]
  })

  #Predict Anomaly
  prediction = model.predict(input_data)[0]

  #Get anomaly score
  score = model.decision_function(input_data)[0]

  #Convert Prediction to readable result
  if prediction == 1:
    result = "Normal Transaction"
  else:
    result = "Anomolous Transaction"

  return result, round(score, 4)

**Step-8 : Create Gradio Interface**

In [42]:
with gr.Blocks() as app:
  gr.Markdown("# 🔎Anomaly Detection Using Isolation Forest")
  gr.Markdown("Enter a transaction amount and transaction frequency to determine whether the transaction is **Normal** or **Anomolous**")

  with gr.Row():
    amount_input = gr.Number(
        label = "Transaction Amount",
        value = 100
    )

    frequency_input = gr.Number(
        label = "Transaction Frequency",
        value = 10
    )

  #Create Button
  predict_button = gr.Button("Detect Anomaly")

  #Output section
  result_output = gr.Textbox(
      label = "Prediction"
  )

  score_output = gr.Number(
      label = "Anomaly Score",
  )

  #Visualization
  gr.Markdown("🪲Dataset Visualization")
  plot_output = gr.Plot(
      value = create_plot()
  )

  #Connect button with prediction function
  predict_button.click(
      fn = detect_anomaly,
      inputs = [amount_input, frequency_input],
      outputs = [result_output, score_output]
  )

#Launch
app.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://60e59fb68753a99824.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://60e59fb68753a99824.gradio.live
